# 09. Transformer from scratch: 텐서 shape로 이해하는 decoder

이 노트북은 작은 decoder-only Transformer를 **PyTorch 기본 연산부터 직접 조립**합니다. 완성 모델의 성능보다 각 연산의 입력·출력 shape와 causal 제약을 설명하는 것이 목표입니다.

학습 목표:

- token/position embedding이 합쳐지는 위치를 추적한다.
- Q, K, V와 multi-head 분할의 shape를 검증한다.
- scaled dot-product attention과 causal mask를 직접 구현한다.
- pre-LN residual block과 next-token loss를 연결한다.
- 저장소의 `TinyCausalTransformer`와 forward 계약을 비교한다.

CUDA → MPS → CPU를 자동 선택하며, 어느 장치에서도 수 초 안에 실행되도록 매우 작은 텐서만 사용합니다.


## 전체 데이터 흐름

표기 규칙은 `B=batch`, `T=time`, `D=model_dim`, `H=heads`, `Dh=D/H`, `V=vocab_size`입니다.

```text
token ids [B,T]
  → token embedding + position embedding [B,T,D]
  → Q/K/V [B,H,T,Dh]
  → attention score [B,H,T,T]
  → masked weighted sum [B,T,D]
  → residual + MLP [B,T,D]
  → vocabulary logits [B,T,V]
  → token t의 logit과 token t+1 label로 cross entropy
```

`assert`가 실패하면 다음 셀로 넘어가지 말고 실제 shape를 먼저 출력해 보세요.


In [ ]:
import math
import sys
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "llm_engineering_lab").is_dir():
            return candidate
    raise RuntimeError("저장소 루트 또는 notebooks/에서 JupyterLab을 실행하세요.")


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "src"))
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(7)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
torch.set_default_device(DEVICE)
print(f"root={ROOT}\ntorch={torch.__version__} | {ACCELERATOR.summary()}")

## 1) 입력 token id와 padding mask

실제 토크나이저 대신 작은 정수 vocabulary를 사용해 모델 구조에 집중합니다. 두 번째 문장의 마지막 `0`은 padding입니다. padding은 key로 참조되지 않아야 합니다.


In [ ]:
PAD_ID, VOCAB_SIZE = 0, 16
BATCH_SIZE, SEQ_LEN, MODEL_DIM, NUM_HEADS = 2, 5, 12, 3
HEAD_DIM = MODEL_DIM // NUM_HEADS

input_ids = torch.tensor(
    [[1, 4, 5, 6, 2], [1, 7, 8, 2, PAD_ID]],
    dtype=torch.long,
    device=DEVICE,
)
attention_mask = input_ids.ne(PAD_ID)

assert input_ids.shape == (BATCH_SIZE, SEQ_LEN)
assert attention_mask.dtype == torch.bool
print("input_ids shape:", tuple(input_ids.shape))
print("attention_mask:\n", attention_mask.int())

## 2) token embedding + learned position embedding

token id는 의미를 학습하는 벡터로, 위치 id는 순서를 학습하는 벡터로 바뀝니다. 두 벡터의 차원이 같으므로 원소별로 더할 수 있습니다. 위치 정보가 없다면 self-attention 자체는 token 순서를 구분하지 못합니다.


In [ ]:
token_embedding = nn.Embedding(VOCAB_SIZE, MODEL_DIM, padding_idx=PAD_ID)
position_embedding = nn.Embedding(SEQ_LEN, MODEL_DIM)
position_ids = torch.arange(SEQ_LEN, device=DEVICE).unsqueeze(0).expand(BATCH_SIZE, -1)

token_vectors = token_embedding(input_ids)
position_vectors = position_embedding(position_ids)
hidden = token_vectors + position_vectors

assert token_vectors.shape == (BATCH_SIZE, SEQ_LEN, MODEL_DIM)
assert position_vectors.shape == hidden.shape
print(
    "token / position / hidden:",
    token_vectors.shape,
    position_vectors.shape,
    hidden.shape,
)
print("padding embedding L2 norm:", token_embedding.weight[PAD_ID].norm().item())

## 3) 하나의 선형층에서 Q, K, V 만들기

실무 구현은 세 선형층 대신 `D → 3D` projection 하나를 사용한 뒤 마지막 축을 셋으로 나누는 경우가 많습니다. 그 다음 `D`를 `H × Dh`로 바꾸고 head 축을 앞으로 옮깁니다.


In [ ]:
qkv_projection = nn.Linear(MODEL_DIM, 3 * MODEL_DIM, bias=False)
packed_qkv = qkv_projection(hidden)
query, key, value = packed_qkv.chunk(3, dim=-1)


def split_heads(tensor: torch.Tensor) -> torch.Tensor:
    batch, time, model_dim = tensor.shape
    return tensor.view(batch, time, NUM_HEADS, HEAD_DIM).transpose(1, 2)


query, key, value = map(split_heads, (query, key, value))
expected = (BATCH_SIZE, NUM_HEADS, SEQ_LEN, HEAD_DIM)
assert query.shape == key.shape == value.shape == expected
print("packed QKV:", tuple(packed_qkv.shape))
print("split Q/K/V:", tuple(query.shape))

## 4) scaled dot-product attention score

각 query와 모든 key의 내적을 계산하면 `[B,H,T,T]`가 됩니다. `sqrt(Dh)`로 나누지 않으면 head dimension이 커질수록 내적 분산이 커져 softmax가 지나치게 뾰족해질 수 있습니다.


In [ ]:
raw_scores = query @ key.transpose(-2, -1)
scaled_scores = raw_scores / math.sqrt(HEAD_DIM)

assert scaled_scores.shape == (BATCH_SIZE, NUM_HEADS, SEQ_LEN, SEQ_LEN)
ratio = raw_scores.std().item() / max(scaled_scores.std().item(), 1e-12)
print("attention score shape:", tuple(scaled_scores.shape))
print(f"std ratio(raw/scaled)={ratio:.3f}, expected~sqrt(Dh)={math.sqrt(HEAD_DIM):.3f}")

## 5) causal mask와 padding mask

decoder LM의 위치 `t`는 `t+1` 이후 token을 볼 수 없어야 합니다. 아래 삼각 mask는 미래 key를 차단하고, padding mask는 실제 문장에 없는 key를 차단합니다. mask 적용 후 softmax 확률에서 미래 영역은 정확히 0이어야 합니다.


In [ ]:
causal_mask = torch.tril(torch.ones(SEQ_LEN, SEQ_LEN, dtype=torch.bool, device=DEVICE))
allowed = causal_mask.view(1, 1, SEQ_LEN, SEQ_LEN)
allowed = allowed & attention_mask.view(BATCH_SIZE, 1, 1, SEQ_LEN)

masked_scores = scaled_scores.masked_fill(
    ~allowed, torch.finfo(scaled_scores.dtype).min
)
attention_weights = F.softmax(masked_scores, dim=-1)

assert attention_weights.shape == scaled_scores.shape
assert torch.allclose(
    torch.triu(attention_weights, diagonal=1), torch.zeros_like(attention_weights)
)
assert torch.allclose(
    attention_weights.sum(dim=-1), torch.ones_like(attention_weights.sum(dim=-1))
)
print("causal mask:\n", causal_mask.int())
print(
    "head 0 attention weights (sample 0):\n", attention_weights[0, 0].round(decimals=3)
)

**관찰 포인트**

- score의 마지막 두 축에서 앞의 `T`는 query 위치, 뒤의 `T`는 key 위치입니다.
- padding query 자체를 0으로 만들지 않아도, padding 위치의 loss를 제외하면 학습 목표에는 포함되지 않습니다.
- 모든 key가 가려진 행을 만들면 softmax가 불안정해질 수 있으므로 실제 batch 계약을 명확히 해야 합니다.

**TODO 09-A:** `causal_mask`를 잠시 모두 `True`로 바꾼 뒤 미래 token을 수정했을 때 이전 위치 logit이 달라지는지 실험하세요.


## 6) multi-head attention 모듈로 묶기

이제 projection, head 분할, mask, weighted sum, output projection을 하나의 `nn.Module`로 묶습니다. 학습을 해부하기 위해 attention probability도 함께 반환합니다.


In [ ]:
class ScratchCausalSelfAttention(nn.Module):
    def __init__(self, model_dim: int, num_heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        if model_dim % num_heads:
            raise ValueError("model_dim은 num_heads로 나누어져야 합니다.")
        self.num_heads = num_heads
        self.head_dim = model_dim // num_heads
        self.qkv = nn.Linear(model_dim, 3 * model_dim)
        self.out = nn.Linear(model_dim, model_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self, x: torch.Tensor, mask: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        batch, time, model_dim = x.shape
        query, key, value = self.qkv(x).chunk(3, dim=-1)

        def heads(tensor: torch.Tensor) -> torch.Tensor:
            return tensor.view(batch, time, self.num_heads, self.head_dim).transpose(
                1, 2
            )

        query, key, value = map(heads, (query, key, value))
        scores = query @ key.transpose(-2, -1) / math.sqrt(self.head_dim)
        causal = torch.tril(torch.ones(time, time, dtype=torch.bool, device=x.device))
        allowed_keys = causal[None, None] & mask[:, None, None, :].bool()
        scores = scores.masked_fill(~allowed_keys, torch.finfo(scores.dtype).min)
        weights = self.dropout(F.softmax(scores, dim=-1))
        context = weights @ value
        context = context.transpose(1, 2).contiguous().view(batch, time, model_dim)
        return self.out(context), weights

In [ ]:
attention = ScratchCausalSelfAttention(MODEL_DIM, NUM_HEADS)
attention_output, observed_weights = attention(hidden, attention_mask)

assert attention_output.shape == hidden.shape
assert observed_weights.shape == (BATCH_SIZE, NUM_HEADS, SEQ_LEN, SEQ_LEN)
future_max = torch.triu(observed_weights, diagonal=1).abs().max().detach().item()
assert future_max == 0.0
print("attention output:", tuple(attention_output.shape))
print("future attention max:", future_max)

## 7) pre-LN Transformer block

decoder block은 보통 다음 두 sub-layer로 구성됩니다.

1. `x = x + Attention(LayerNorm(x))`
2. `x = x + MLP(LayerNorm(x))`

residual 경로는 깊은 모델에서 gradient가 흐를 통로를 제공합니다. MLP는 각 token 위치에 독립적으로 적용됩니다.


In [ ]:
class ScratchTransformerBlock(nn.Module):
    def __init__(self, model_dim: int, num_heads: int, ff_dim: int) -> None:
        super().__init__()
        self.attn_norm = nn.LayerNorm(model_dim)
        self.attn = ScratchCausalSelfAttention(model_dim, num_heads)
        self.ff_norm = nn.LayerNorm(model_dim)
        self.ff = nn.Sequential(
            nn.Linear(model_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, model_dim),
        )

    def forward(
        self, x: torch.Tensor, mask: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        attended, weights = self.attn(self.attn_norm(x), mask)
        x = x + attended
        x = x + self.ff(self.ff_norm(x))
        return x, weights


block = ScratchTransformerBlock(MODEL_DIM, NUM_HEADS, ff_dim=24)
block_output, block_weights = block(hidden, attention_mask)
assert block_output.shape == hidden.shape
print("block output:", tuple(block_output.shape))

In [ ]:
# gradient가 embedding과 QKV projection까지 실제로 흐르는지 확인합니다.
diagnostic_loss = block_output.square().mean()
diagnostic_loss.backward()
grad_norm = block.attn.qkv.weight.grad.norm().item()

assert grad_norm > 0
print(f"diagnostic_loss={diagnostic_loss.item():.4f}, qkv_grad_norm={grad_norm:.4f}")

## 8) vocabulary projection과 next-token loss

각 위치의 hidden state를 vocabulary 크기의 logit으로 바꿉니다. teacher forcing에서는 입력과 label을 한 칸 이동합니다.

- `logits[:, 0]`은 `labels[:, 1]`을 예측
- `logits[:, T-2]`는 `labels[:, T-1]`을 예측
- padding label은 loss에서 제외


In [ ]:
class ScratchTinyLM(nn.Module):
    def __init__(self, vocab_size: int, pad_id: int, max_length: int) -> None:
        super().__init__()
        self.pad_id = pad_id
        self.token_embedding = nn.Embedding(vocab_size, MODEL_DIM, padding_idx=pad_id)
        self.position_embedding = nn.Embedding(max_length, MODEL_DIM)
        self.blocks = nn.ModuleList(
            [ScratchTransformerBlock(MODEL_DIM, NUM_HEADS, ff_dim=24)]
        )
        self.norm = nn.LayerNorm(MODEL_DIM)
        self.lm_head = nn.Linear(MODEL_DIM, vocab_size, bias=False)

    def forward(
        self,
        token_ids: torch.Tensor,
        mask: torch.Tensor,
        labels: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        batch, time = token_ids.shape
        positions = torch.arange(time, device=token_ids.device)[None, :].expand(
            batch, -1
        )
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        for block in self.blocks:
            x, _ = block(x, mask)
        logits = self.lm_head(self.norm(x))
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=self.pad_id,
            )
        return logits, loss

In [ ]:
scratch_lm = ScratchTinyLM(VOCAB_SIZE, PAD_ID, SEQ_LEN).to(DEVICE)
optimizer = torch.optim.AdamW(scratch_lm.parameters(), lr=1e-3)
logits, next_token_loss = scratch_lm(input_ids, attention_mask, labels=input_ids)

assert logits.shape == (BATCH_SIZE, SEQ_LEN, VOCAB_SIZE)
assert next_token_loss is not None and torch.isfinite(next_token_loss)
optimizer.zero_grad(set_to_none=True)
next_token_loss.backward()
optimizer.step()
print(
    "logits:", tuple(logits.shape), "next-token loss:", round(next_token_loss.item(), 4)
)

## 9) 저장소 구현과 forward 계약 비교

직접 만든 모델은 개념 확인용입니다. 저장소 구현은 입력 검증, dropout, weight tying, KV cache, generation까지 포함합니다. 내부 수치가 같을 필요는 없지만 `[B,T] → [B,T,V]`와 next-token loss 계약은 같아야 합니다.


In [ ]:
from llm_engineering_lab.transformer import TinyCausalTransformer, TinyTransformerConfig

production_shaped = TinyCausalTransformer(
    TinyTransformerConfig(
        vocab_size=VOCAB_SIZE,
        pad_id=PAD_ID,
        max_seq_length=SEQ_LEN,
        model_dim=MODEL_DIM,
        num_heads=NUM_HEADS,
        num_layers=1,
        feed_forward_dim=24,
        dropout=0.0,
    )
).to(DEVICE)
result = production_shaped(input_ids, attention_mask=attention_mask, labels=input_ids)

assert result.logits.shape == logits.shape
assert result.loss is not None and torch.isfinite(result.loss)
print(
    "repo logits:",
    tuple(result.logits.shape),
    "repo loss:",
    round(result.loss.item(), 4),
)
print("parameter count:", sum(p.numel() for p in production_shaped.parameters()))

## 마무리 도전 과제

- **TODO 09-B (필수):** `ScratchCausalSelfAttention`에 attention dropout을 추가하고 `train()`/`eval()` 결과 차이를 고정 seed로 확인하세요.
- **TODO 09-C (shape):** `B=3, T=7, D=24, H=6`으로 바꾸고 모든 assert가 통과하게 만드세요.
- **TODO 09-D (누수 테스트):** 미래 token만 바꾼 두 입력에서 이전 위치 logits가 동일한지 `torch.testing.assert_close`로 검증하세요.
- **TODO 09-E (심화):** learned position embedding을 sinusoidal 또는 RoPE로 교체하고 장단점을 기록하세요.
- **TODO 09-F (현업):** attention probability 전체를 저장할 때 필요한 메모리를 `B×H×T×T×dtype_bytes`로 계산하세요.

완료 기준: 각 단계의 shape와 causal mask가 필요한 이유를 그림 없이도 설명할 수 있습니다.
